> **교육 목표** 정상성과 ACF/PACF를 이해하고, ARIMA가 이 데이터에서 어디까지 통하는지 확인합니다.
>
> **핵심 내용** ADF 정상성 검정 → 차분 → ACF·PACF → ARIMA(1,1,1) 학습 → 잔차 확인 → Test 기간 예측

In [ ]:
%%html
<style>
table {
    float: left;
    clear: both;
}
</style><!-- Markdown 왼쪽 정렬 -->

# D5_03 정상성 · ACF/PACF · ARIMA

> **오늘 질문:** 과거 값만으로 미래를 예측하는 ARIMA는 이 데이터에서 얼마나 통할까?

| 표시 | 의미 |
|---|---|
| 📋 제공 | 코드가 들어 있습니다. 실행만 하세요 |
| ✍️ 입력 | 설명을 보고 직접 입력하세요 |
| 💬 해석 | 결과를 보고 답을 적어 보세요 |

## 0. 준비 📋

모든 모델은 **앞 80%(Train)로만** 학습합니다. 뒤 20%(Test, 10월 20일 ~ 12월 31일)는 평가용입니다.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")      # 모형 수렴 관련 경고 숨기기
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
plt.rcParams["axes.unicode_minus"] = False
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA

# 1교시에서 한 시간 정리 (00:00 기록 바로잡기 → 구간 시작 시각으로)
se = pd.read_csv("../../data/steel_energy.csv")
se["date"] = pd.to_datetime(se["date"], format="%d/%m/%Y %H:%M")
midnight = (se["date"].dt.hour == 0) & (se["date"].dt.minute == 0)
se.loc[midnight, "date"] += pd.Timedelta(days=1)
se["date"] -= pd.Timedelta(minutes=15)
se = se.set_index("date").sort_index()
daily = se["usage_kwh"].resample("D").sum()

cut = int(len(daily) * 0.8)
train, test = daily.iloc[:cut], daily.iloc[cut:]
print(f"Train {train.index[0].date()} ~ {train.index[-1].date()} ({len(train)}일) / Test {test.index[0].date()} ~ ({len(test)}일)")

## 1. 정상성 검정 (ADF) ✍️

귀무가설은 "비정상이다"입니다. p-value가 0.05보다 작으면 정상으로 봅니다. `adfuller()` 결과의 두 번째 값(`[1]`)이 p-value입니다.

In [ ]:
# 힌트: adfuller(시리즈)[1]

## 2. 차분 전후 그래프 ✍️

In [ ]:
# 힌트: 시리즈.diff()

## 3. ACF / PACF ✍️

차분한 데이터로 그립니다. 파란 띠 밖으로 나간 막대만 의미 있는 관계입니다.

In [ ]:
# 힌트: plot_acf(시리즈, lags=28), plot_pacf(...)

**Q1.** ACF에서 특히 크게 튀는 시차(lag)는 몇인가요? 무엇을 뜻할까요?

> 답:


## 4. ARIMA(1,1,1) 학습 ✍️

어제 값 1개(p=1), 차분 1번(d=1), 어제 오차 1개(q=1)를 쓰는 모형입니다.

> 변수 이름: `arima`

In [ ]:
# 힌트: ARIMA(시리즈, order=(1, 1, 1)).fit()

## 5. 잔차 확인 ✍️

좋은 모형이라면 잔차에 남은 패턴이 없어야 합니다. 잔차의 ACF를 봅니다.

In [ ]:
# 힌트: 모형.resid 는 잔차

## 6. Test 기간 한 번에 예측 📋

Test 73일을 한 번에 예측해 실제와 비교합니다. (매주 새로 예측하는 공정한 평가는 5교시에)

In [ ]:
forecast = arima.forecast(len(test))
plt.figure(figsize=(10, 3.5))
plt.plot(test.index, test, label="실제")
plt.plot(test.index, forecast, label="ARIMA(1,1,1)")
plt.legend()
plt.title("ARIMA 73일 예측")
plt.show()

## 정리 💬

**Q2.** 잔차의 ACF와 73일 예측 그래프를 보고, ARIMA(1,1,1)이 놓친 것은 무엇인지 설명해 보세요.

> 답:
